# Task 4 — Build a Simple RAG System (PDF / Excel knowledge base)

**Goal:** answer questions from a **PDF or Excel sheet** using Retrieval-Augmented Generation = **retriever + generator**.

Pipeline (same as the lecture's *Data Workflow*):

`Load data → Chunking → Embedding → Vector DB (FAISS) → Query embedding → Semantic search → Prompt (query + chunks) → LLM → Answer`

| Part | Tool used |
|---|---|
| Loader | `pypdf` (PDF) / `pandas` (Excel, CSV) |
| Chunking | fixed-size word chunks with overlap (PDF) / one row = one chunk (Excel) |
| Embedding model | `all-MiniLM-L6-v2` (384-dim) |
| Vector DB | FAISS |
| Generator (LLM) | `Qwen2.5-1.5B-Instruct` |

> **Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**.
> **Knowledge base:** upload your provided PDF/Excel file to Colab (left sidebar → Files → Upload) *before* running Step 2. If no file is found, the notebook creates a small sample Excel sheet so you can still see it working.

## 1️⃣ Install & imports

In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf openpyxl transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import os, re, glob
import numpy as np
import pandas as pd
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 2️⃣ Choose the knowledge base (PDF or Excel)

Set `KB_PATH` to your file, e.g. `"/content/DS.pdf"` or `"/content/project_managment.xlsx"`.
Leave it as `None` to auto-detect the first PDF/Excel/CSV in the current folder or `/content`.

In [3]:
KB_PATH = None   # e.g. "/content/DS.pdf"  or  "/content/project_managment.xlsx"

def find_file():
    for folder in [".", "/content"]:
        for ext in ("pdf", "xlsx", "xls", "csv"):
            found = sorted(glob.glob(os.path.join(folder, f"*.{ext}")))
            if found:
                return found[0]
    return None

def make_sample_excel(path="sample_project_management.xlsx"):
    """Fallback so the notebook always runs: a small project-management sheet."""
    rows = [
        ("Marketing", "Market Research",       "Alice", "2024-01-01", 13, "2024-01-14", "78%"),
        ("Marketing", "Brand Design",          "Omar",  "2024-01-10", 20, "2024-01-30", "55%"),
        ("Software",  "Software Development",  "Sara",  "2024-01-05", 45, "2024-02-19", "62%"),
        ("Software",  "Quality Assurance",     "Youssef","2024-02-10", 15, "2024-02-25", "30%"),
        ("Software",  "Deployment",            "Mona",  "2024-02-26",  5, "2024-03-02", "0%"),
        ("HR",        "Recruit Data Engineers","Hana",  "2024-01-02", 30, "2024-02-01", "100%"),
        ("HR",        "Employee Onboarding",   "Khaled","2024-02-05", 10, "2024-02-15", "100%"),
        ("Finance",   "Budget Planning",       "Laila", "2024-01-03", 12, "2024-01-15", "100%"),
        ("Finance",   "Quarterly Audit",       "Tamer", "2024-03-01", 25, "2024-03-26", "10%"),
        ("Support",   "Customer FAQ Update",   "Nour",  "2024-01-20",  8, "2024-01-28", "90%"),
    ]
    cols = ["Project", "Task", "Assigned To", "Start Date", "Days Required", "End Date", "Progress"]
    pd.DataFrame(rows, columns=cols).to_excel(path, index=False)
    return path

if KB_PATH is None:
    KB_PATH = find_file()
if KB_PATH is None:
    KB_PATH = make_sample_excel()
    print("No file found -> created a sample Excel sheet.")

print("Knowledge base file:", KB_PATH)

No file found -> created a sample Excel sheet.
Knowledge base file: sample_project_management.xlsx


## 3️⃣ Load the document

- **PDF:** extract the text page by page.
- **Excel/CSV:** convert every row into a readable sentence (`Column: value | Column: value ...`) so the embedding model understands it.

In [4]:
def load_pdf(path):
    pages = []
    for i, page in enumerate(PdfReader(path).pages, 1):
        text = (page.extract_text() or "").strip()
        if text:
            pages.append({"page": i, "text": text})
    return pages

def load_table(path):
    df = pd.read_csv(path) if path.lower().endswith(".csv") else pd.read_excel(path)
    df = df.fillna("")
    df.columns = [str(c).strip() for c in df.columns]
    rows = []
    for i, row in df.iterrows():
        parts = [f"{c}: {str(row[c]).strip()}" for c in df.columns if str(row[c]).strip()]
        if parts:
            rows.append({"row": i + 2, "text": " | ".join(parts)})   # +2 = Excel row number (header is row 1)
    return rows, df

ext = os.path.splitext(KB_PATH)[1].lower()

if ext == ".pdf":
    docs = load_pdf(KB_PATH)
    print(f"PDF loaded: {len(docs)} pages, {sum(len(d['text'].split()) for d in docs)} words")
    print("\nSample:\n", docs[0]["text"][:400])
else:
    docs, df = load_table(KB_PATH)
    print(f"Table loaded: {len(docs)} rows | columns: {list(df.columns)}")
    print("\nSample rows:")
    for d in docs[:2]:
        print(" -", d["text"])

Table loaded: 10 rows | columns: ['Project', 'Task', 'Assigned To', 'Start Date', 'Days Required', 'End Date', 'Progress']

Sample rows:
 - Project: Marketing | Task: Market Research | Assigned To: Alice | Start Date: 2024-01-01 | Days Required: 13 | End Date: 2024-01-14 | Progress: 78%
 - Project: Marketing | Task: Brand Design | Assigned To: Omar | Start Date: 2024-01-10 | Days Required: 20 | End Date: 2024-01-30 | Progress: 55%


## 4️⃣ Chunking

- **PDF:** chunks of ~120 words with 20 words overlap (overlap protects information at chunk boundaries).
- **Excel:** every row is already a small meaningful unit, so **one row = one chunk** (keeps a task's information together).

Each chunk keeps its **source** (page or row number) so the answer can cite where it came from.

In [5]:
CHUNK_SIZE, OVERLAP = 120, 20

def chunk_pdf(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    chunks = []
    for p in pages:
        words = re.sub(r"\s+", " ", p["text"]).split()
        for i in range(0, len(words), chunk_size - overlap):
            piece = " ".join(words[i:i + chunk_size])
            if len(piece) > 30:
                chunks.append({"text": piece, "source": f"page {p['page']}"})
            if i + chunk_size >= len(words):
                break
    return chunks

def chunk_rows(rows):
    return [{"text": r["text"], "source": f"row {r['row']}"} for r in rows]

chunks = chunk_pdf(docs) if ext == ".pdf" else chunk_rows(docs)
print("Total chunks:", len(chunks))
print("\nSample chunk:", chunks[0])

Total chunks: 10

Sample chunk: {'text': 'Project: Marketing | Task: Market Research | Assigned To: Alice | Start Date: 2024-01-01 | Days Required: 13 | End Date: 2024-01-14 | Progress: 78%', 'source': 'row 2'}


## 5️⃣ Embedding + Vector database (FAISS)

Each chunk becomes a 384-dim vector. We normalize the vectors and use an inner-product index, which is equivalent to **cosine similarity**.

In [6]:
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

embeddings = embed_model.encode(
    [c["text"] for c in chunks],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print("Embeddings shape:", embeddings.shape, "| vectors in FAISS:", index.ntotal)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (10, 384) | vectors in FAISS: 10


## 6️⃣ Retriever (semantic search)

In [7]:
def retrieve(query, k=3):
    q = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q, min(k, index.ntotal))
    return [{**chunks[i], "score": float(s)} for s, i in zip(scores[0], ids[0])]

# quick test of the retriever alone
test_q = "What is the main topic of this document?"
for r in retrieve(test_q):
    print(f"[{r['score']:.3f}] ({r['source']}) {r['text'][:150]}...")

[0.238] (row 2) Project: Marketing | Task: Market Research | Assigned To: Alice | Start Date: 2024-01-01 | Days Required: 13 | End Date: 2024-01-14 | Progress: 78%...
[0.229] (row 10) Project: Finance | Task: Quarterly Audit | Assigned To: Tamer | Start Date: 2024-03-01 | Days Required: 25 | End Date: 2024-03-26 | Progress: 10%...
[0.215] (row 3) Project: Marketing | Task: Brand Design | Assigned To: Omar | Start Date: 2024-01-10 | Days Required: 20 | End Date: 2024-01-30 | Progress: 55%...


## 7️⃣ Generator (LLM)

`Qwen2.5-1.5B-Instruct` is small enough for a free T4 GPU and follows instructions well. Greedy decoding makes the answers stable and easy to compare.

In [8]:
LLM_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
llm.eval()

@torch.no_grad()
def generate(messages, max_new_tokens=200):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("LLM ready on", llm.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM ready on cuda:0


## 8️⃣ The RAG function (retrieve → augment → generate)

Includes three ideas from the lecture:
- **Grounded answers:** the model must answer *only* from the retrieved context.
- **Fallback:** if retrieval is weak (`score < MIN_SCORE`) the system says it doesn't know instead of guessing.
- **Explainability:** the answer comes with its sources (page/row).

In [9]:
MIN_SCORE = 0.20   # below this the retrieved text is considered unrelated

def rag_answer(question, k=3, max_new_tokens=200):
    hits = retrieve(question, k=k)

    if hits[0]["score"] < MIN_SCORE:
        return "I don't know based on the provided document.", hits

    context = "\n".join(f"[{i}] ({h['source']}) {h['text']}" for i, h in enumerate(hits, 1))
    messages = [
        {"role": "system", "content":
            "You answer questions using ONLY the provided context. "
            "If the answer is not in the context, say: 'I don't know based on the provided document.' "
            "Be concise and mention the source (page/row) you used."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    return generate(messages, max_new_tokens), hits

def plain_llm_answer(question, max_new_tokens=200):
    """Answer WITHOUT retrieval (for the before/after comparison)."""
    return generate([{"role": "user", "content": question}], max_new_tokens)

def ask(question, k=3):
    print("=" * 90)
    print("QUESTION:", question)
    print("\n--- WITHOUT RAG ---")
    print(plain_llm_answer(question))
    answer, hits = rag_answer(question, k=k)
    print("\n--- RETRIEVED CHUNKS ---")
    for h in hits:
        print(f"[{h['score']:.3f}] ({h['source']}) {h['text'][:200]}")
    print("\n--- WITH RAG ---")
    print(answer)

## 9️⃣ Test the system

**Edit `questions` to match your file** (the default ones fit the sample project-management sheet).
For a PDF such as `DS.pdf`, try e.g. *"What is Data Science?"*, *"What are the steps of the data science lifecycle?"*, and one question that is **not** in the document to check the fallback.

In [10]:
questions = [
    "Who is assigned to Market Research?",
    "What is the progress of the Software Development task?",
    "How many days does Quality Assurance take and who is assigned to it?",
    "Who is the CEO of the company?",     # not in the document -> should say it doesn't know
]

for q in questions:
    ask(q)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QUESTION: Who is assigned to Market Research?

--- WITHOUT RAG ---
Market research is typically assigned to someone with expertise in market analysis and data collection. This person may be an analyst or researcher who has experience working with large datasets and conducting surveys or focus groups.

In some cases, the role of market research might also involve coordinating with other departments within an organization, such as sales or product development, to ensure that the insights gained from the research are being used effectively to inform business decisions.

The specific responsibilities of a market research team can vary depending on the size and scope of the project, but generally include tasks such as:

1. Defining research objectives and hypotheses
2. Developing research methods and tools 
3. Collecting and analyzing data
4. Interpreting results and drawing conclusions
5. Presenting findings to stakeholders

Overall, the goal of market research is to gain a deeper understa

## 🔟 Interactive mode (optional)

In [11]:
while True:
    q = input("Ask a question (type 'exit' to stop): ").strip()
    if q.lower() in ("exit", "quit", ""):
        break
    answer, hits = rag_answer(q)
    print("\nAnswer:", answer)
    print("Sources:", ", ".join(h["source"] for h in hits), "\n")

Ask a question (type 'exit' to stop):  what is the name of company



Answer: I don't know based on the provided document.
Sources: row 3, row 5, row 2 



Ask a question (type 'exit' to stop):  how are the persons in HR department



Answer: Based on the provided information, there are three people assigned to tasks within the HR department:

1. **Khaled** - Assigned to "Employee Onboarding" with a start date of February 5, 2024, an end date of February 15, 2024, and a progress of 100%.
2. **Hana** - Assigned to "Recruit Data Engineers" with a start date of January 2, 2024, an end date of February 1, 2024, and a progress of 100%.
3. **Youssef** - Assigned to "Quality Assurance" with a start date of February 10, 2024, an end date of February 25, 2024, and a progress of 30%.
Sources: row 8, row 7, row 5 



Ask a question (type 'exit' to stop):  exit
